# CLI Surface Proof

This notebook proves the first operational surface through the actual `onto-canon6` CLI.

It uses a deterministic extractor stand-in so the workflow can be inspected locally without relying on a live model call.

In [1]:
from __future__ import annotations

import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / "src").exists() and PROJECT_ROOT.parent != PROJECT_ROOT:
    PROJECT_ROOT = PROJECT_ROOT.parent

if not (PROJECT_ROOT / "src").exists():
    raise RuntimeError("Could not locate onto-canon6 repo root from notebook cwd")

for candidate_path in (PROJECT_ROOT / "src", PROJECT_ROOT.parent / "llm_client"):
    if candidate_path.exists():
        candidate_str = str(candidate_path)
        if candidate_str not in sys.path:
            sys.path.insert(0, candidate_str)

import contextlib
import io
import json
from tempfile import TemporaryDirectory

from onto_canon6 import cli as cli_module
from onto_canon6.pipeline import (
    CandidateAssertionImport,
    EvidenceSpan,
    ProfileRef,
    ReviewService,
    SourceArtifactRef,
)


In [2]:
class DeterministicTextExtractionService:
    """Provide a local stand-in for the remote extraction boundary."""

    def __init__(self, *, review_service: ReviewService) -> None:
        self._review_service = review_service

    def extract_and_submit(
        self,
        *,
        source_text: str,
        profile_id: str,
        profile_version: str,
        submitted_by: str,
        source_ref: str,
        source_kind: str = "text_file",
        source_label: str | None = None,
        source_metadata: dict[str, object] | None = None,
    ) -> tuple[object, ...]:
        del source_metadata
        phrase = "aligned messaging"
        start_char = source_text.index(phrase)
        end_char = start_char + len(phrase)
        candidate_import = CandidateAssertionImport(
            profile=ProfileRef(profile_id=profile_id, profile_version=profile_version),
            payload={
                "predicate": "oc:signals_alignment",
                "roles": {
                    "subject": [{"kind": "value", "value_kind": "string", "value": "Campaign Alpha"}],
                    "object": [{"kind": "value", "value_kind": "string", "value": phrase}],
                },
            },
            submitted_by=submitted_by,
            source_artifact=SourceArtifactRef(
                source_kind=source_kind,
                source_ref=source_ref,
                source_label=source_label,
                source_metadata={},
                content_text=source_text,
            ),
            evidence_spans=(EvidenceSpan(start_char=start_char, end_char=end_char, text=phrase),),
            claim_text="Campaign Alpha used aligned messaging.",
        )
        return (self._review_service.submit_candidate_import(candidate_import=candidate_import),)

def run_cli(args: list[str]) -> tuple[int, str, str]:
    stdout_buffer = io.StringIO()
    stderr_buffer = io.StringIO()
    with contextlib.redirect_stdout(stdout_buffer), contextlib.redirect_stderr(stderr_buffer):
        exit_code = cli_module.main(args)
    return exit_code, stdout_buffer.getvalue(), stderr_buffer.getvalue()


In [3]:
tmp_dir = TemporaryDirectory()
workspace = Path(tmp_dir.name)
source_path = workspace / "source.txt"
source_path.write_text("Campaign Alpha used aligned messaging across channels.", encoding="utf-8")
review_db_path = workspace / "review.sqlite3"
overlay_root = workspace / "ontology_overlays"

original_extractor = cli_module.TextExtractionService
cli_module.TextExtractionService = DeterministicTextExtractionService


In [4]:
extract_exit, extract_stdout, extract_stderr = run_cli([
    "extract-text",
    "--input", str(source_path),
    "--profile-id", "psyop_seed",
    "--profile-version", "0.1.0",
    "--submitted-by", "analyst:notebook",
    "--review-db-path", str(review_db_path),
    "--overlay-root", str(overlay_root),
    "--output", "json",
])
assert extract_exit == 0, extract_stderr
extract_result = json.loads(extract_stdout)
extract_result


[{'candidate': {'candidate_id': 'cand_951275bc3cf8497eb679ff65',
   'claim_text': 'Campaign Alpha used aligned messaging.',
   'evidence_spans': [{'end_char': 37,
     'start_char': 20,
     'text': 'aligned messaging'}],
   'normalized_payload': {'predicate': 'oc:signals_alignment',
    'roles': {'object': [{'kind': 'value',
       'value': 'aligned messaging',
       'value_kind': 'string'}],
     'subject': [{'kind': 'value',
       'value': 'Campaign Alpha',
       'value_kind': 'string'}]}},
   'payload': {'predicate': 'oc:signals_alignment',
    'roles': {'object': [{'kind': 'value',
       'value': 'aligned messaging',
       'value_kind': 'string'}],
     'subject': [{'kind': 'value',
       'value': 'Campaign Alpha',
       'value_kind': 'string'}]}},
   'payload_hash': 'sha256:61808a0228906bbaaf1280423471693fa04fa39e993210adeea8b53d4f7e0b71',
   'profile': {'profile_id': 'psyop_seed', 'profile_version': '0.1.0'},
   'proposal_ids': ['prop_22e13cd665e9fe60f13aad1e'],
   'prove

In [5]:
candidate_id = extract_result[0]["candidate"]["candidate_id"]
proposal_id = extract_result[0]["proposals"][0]["proposal_id"]

list_exit, list_stdout, list_stderr = run_cli([
    "list-candidates",
    "--review-db-path", str(review_db_path),
    "--output", "json",
])
assert list_exit == 0, list_stderr
json.loads(list_stdout)


[{'candidate_id': 'cand_951275bc3cf8497eb679ff65',
  'claim_text': 'Campaign Alpha used aligned messaging.',
  'evidence_spans': [{'end_char': 37,
    'start_char': 20,
    'text': 'aligned messaging'}],
  'normalized_payload': {'predicate': 'oc:signals_alignment',
   'roles': {'object': [{'kind': 'value',
      'value': 'aligned messaging',
      'value_kind': 'string'}],
    'subject': [{'kind': 'value',
      'value': 'Campaign Alpha',
      'value_kind': 'string'}]}},
  'payload': {'predicate': 'oc:signals_alignment',
   'roles': {'object': [{'kind': 'value',
      'value': 'aligned messaging',
      'value_kind': 'string'}],
    'subject': [{'kind': 'value',
      'value': 'Campaign Alpha',
      'value_kind': 'string'}]}},
  'payload_hash': 'sha256:61808a0228906bbaaf1280423471693fa04fa39e993210adeea8b53d4f7e0b71',
  'profile': {'profile_id': 'psyop_seed', 'profile_version': '0.1.0'},
  'proposal_ids': ['prop_22e13cd665e9fe60f13aad1e'],
  'provenance': {'source_artifact': {'conten

In [6]:
review_candidate_exit, review_candidate_stdout, review_candidate_stderr = run_cli([
    "review-candidate",
    "--candidate-id", candidate_id,
    "--decision", "accepted",
    "--actor-id", "analyst:reviewer",
    "--review-db-path", str(review_db_path),
    "--output", "json",
])
assert review_candidate_exit == 0, review_candidate_stderr
json.loads(review_candidate_stdout)


{'candidate_id': 'cand_951275bc3cf8497eb679ff65',
 'claim_text': 'Campaign Alpha used aligned messaging.',
 'evidence_spans': [{'end_char': 37,
   'start_char': 20,
   'text': 'aligned messaging'}],
 'normalized_payload': {'predicate': 'oc:signals_alignment',
  'roles': {'object': [{'kind': 'value',
     'value': 'aligned messaging',
     'value_kind': 'string'}],
   'subject': [{'kind': 'value',
     'value': 'Campaign Alpha',
     'value_kind': 'string'}]}},
 'payload': {'predicate': 'oc:signals_alignment',
  'roles': {'object': [{'kind': 'value',
     'value': 'aligned messaging',
     'value_kind': 'string'}],
   'subject': [{'kind': 'value',
     'value': 'Campaign Alpha',
     'value_kind': 'string'}]}},
 'payload_hash': 'sha256:61808a0228906bbaaf1280423471693fa04fa39e993210adeea8b53d4f7e0b71',
 'profile': {'profile_id': 'psyop_seed', 'profile_version': '0.1.0'},
 'proposal_ids': ['prop_22e13cd665e9fe60f13aad1e'],
 'provenance': {'source_artifact': {'content_text': 'Campaign Alph

In [7]:
review_proposal_exit, review_proposal_stdout, review_proposal_stderr = run_cli([
    "review-proposal",
    "--proposal-id", proposal_id,
    "--decision", "accepted",
    "--actor-id", "analyst:reviewer",
    "--acceptance-policy", "apply_to_overlay",
    "--review-db-path", str(review_db_path),
    "--output", "json",
])
assert review_proposal_exit == 0, review_proposal_stderr
json.loads(review_proposal_stdout)


{'application_status': 'pending_overlay_apply',
 'candidate_ids': ['cand_951275bc3cf8497eb679ff65'],
 'created_at': '2026-03-18T02:58:14.641122Z',
 'details': {'candidate_source_kind': 'text_file',
  'source': 'validate_assertion_payload'},
 'overlay_application': None,
 'profile': {'profile_id': 'psyop_seed', 'profile_version': '0.1.0'},
 'proposal_id': 'prop_22e13cd665e9fe60f13aad1e',
 'proposal_key': 'sha256:22e13cd665e9fe60f13aad1e953d210364351edb2239969e344895ea2afaab97',
 'proposal_kind': 'predicate',
 'proposed_value': 'oc:signals_alignment',
 'reason': "default ontology mode 'mixed' resolved action 'propose' for predicate",
 'review': {'acceptance_policy': 'apply_to_overlay',
  'actor_id': 'analyst:reviewer',
  'application_status': 'pending_overlay_apply',
  'created_at': '2026-03-18T02:58:14.677840Z',
  'decision': 'accepted',
  'decision_id': 'dec_ce477aaf4c5e41e7aef9d0a5',
  'note_text': None,
  'proposal_id': 'prop_22e13cd665e9fe60f13aad1e'},
 'status': 'accepted',
 'targe

In [8]:
apply_exit, apply_stdout, apply_stderr = run_cli([
    "apply-overlay",
    "--proposal-id", proposal_id,
    "--actor-id", "analyst:reviewer",
    "--review-db-path", str(review_db_path),
    "--overlay-root", str(overlay_root),
    "--output", "json",
])
assert apply_exit == 0, apply_stderr
apply_result = json.loads(apply_stdout)
apply_result


{'application_id': 'oapp_396754b47e2845d89b477e89',
 'applied_by': 'analyst:reviewer',
 'applied_value': 'oc:signals_alignment',
 'content_path': '/tmp/tmpjqi2khdu/ontology_overlays/onto_canon_psyop_seed__overlay/0.1.0/predicate_additions.jsonl',
 'created_at': '2026-03-18T02:58:14.690605Z',
 'overlay_pack': {'pack_id': 'onto_canon_psyop_seed__overlay',
  'pack_version': '0.1.0'},
 'profile': {'profile_id': 'psyop_seed', 'profile_version': '0.1.0'},
 'proposal_id': 'prop_22e13cd665e9fe60f13aad1e',
 'proposal_kind': 'predicate'}

In [9]:
proposal_list_exit, proposal_list_stdout, proposal_list_stderr = run_cli([
    "list-proposals",
    "--review-db-path", str(review_db_path),
    "--status", "accepted",
    "--output", "json",
])
assert proposal_list_exit == 0, proposal_list_stderr
overlay_content_path = Path(apply_result["content_path"])
{
    "accepted_proposals": json.loads(proposal_list_stdout),
    "overlay_content": overlay_content_path.read_text(encoding="utf-8"),
}


{'accepted_proposals': [{'application_status': 'applied_to_overlay',
   'candidate_ids': ['cand_951275bc3cf8497eb679ff65'],
   'created_at': '2026-03-18T02:58:14.641122Z',
   'details': {'candidate_source_kind': 'text_file',
    'source': 'validate_assertion_payload'},
   'overlay_application': {'application_id': 'oapp_396754b47e2845d89b477e89',
    'applied_by': 'analyst:reviewer',
    'applied_value': 'oc:signals_alignment',
    'content_path': '/tmp/tmpjqi2khdu/ontology_overlays/onto_canon_psyop_seed__overlay/0.1.0/predicate_additions.jsonl',
    'created_at': '2026-03-18T02:58:14.690605Z',
    'overlay_pack': {'pack_id': 'onto_canon_psyop_seed__overlay',
     'pack_version': '0.1.0'},
    'profile': {'profile_id': 'psyop_seed', 'profile_version': '0.1.0'},
    'proposal_id': 'prop_22e13cd665e9fe60f13aad1e',
    'proposal_kind': 'predicate'},
   'profile': {'profile_id': 'psyop_seed', 'profile_version': '0.1.0'},
   'proposal_id': 'prop_22e13cd665e9fe60f13aad1e',
   'proposal_key': 

In [10]:
cli_module.TextExtractionService = original_extractor
tmp_dir.cleanup()
